# Exploring Bartons' internal eager kernels

These direct bindings exist for development, parity testing, and inspection while Bartons transitions toward an expression-only public API. They are not recommended for application code. Use `bartons.indicators` in DataFrame and LazyFrame queries.

In [1]:
import polars as pl

from bartons import kernels
from bartons.samples import sample_prices

## Load the sample prices

In [2]:
prices = sample_prices("daily", max_bars=250)
close = prices["close"]
high = prices["high"]
low = prices["low"]
prices.head()

date,open,high,low,close,volume
date,f64,f64,f64,f64,i64
2023-08-14,177.268901,178.982126,176.611497,178.753036,43675600
2023-08-15,178.175305,178.772932,176.352513,176.750931,43622600
2023-08-16,176.432212,177.836645,175.804689,175.87442,46964900
2023-08-17,176.442174,176.810712,172.796589,173.314545,66062900
2023-08-18,171.621237,174.41021,171.28258,173.802612,61114200


## Scalar-output kernels

Kernel parameters are keyword-only. Each call returns a Series that can be renamed and assembled into a DataFrame.

In [3]:
scalar = pl.DataFrame(
    {
        "date": prices["date"],
        "close": close,
        "ema20": kernels.ema(close, period=20),
        "rsi14": kernels.rsi(close, period=14),
        "atr14": kernels.atr(high, low, close, period=14),
    }
)
scalar.tail()

date,close,ema20,rsi14,atr14
date,f64,f64,f64,f64
2024-08-05,209.270004,219.648137,38.022893,6.739317
2024-08-06,207.229996,218.465457,36.137478,6.89508
2024-08-07,209.820007,217.642081,40.192313,6.920431
2024-08-08,213.309998,217.229501,45.237928,6.809686
2024-08-09,216.240005,217.135264,49.11892,6.666851


## Struct-output kernels

Fused multi-output kernels return one struct Series. Put it in a DataFrame and unnest it to obtain ordinary columns.

In [4]:
structures = pl.DataFrame(
    {
        "date": prices["date"],
        "dmi": kernels.dmi(high, low, close, period=14),
        "aroon": kernels.aroon(high, low, period=14),
    }
)
structures.unnest("dmi", "aroon").tail()

date,adx,pdi,mdi,aroondown,aroonup
date,f64,f64,f64,f64,f64
2024-08-05,27.150618,14.432837,38.54906,100.0,0.0
2024-08-06,28.462564,13.099165,34.986918,92.857143,0.0
2024-08-07,28.869322,15.886258,32.368841,85.714286,0.0
2024-08-08,29.124991,15.578824,30.545592,78.571429,7.142857
2024-08-09,28.800148,17.540192,28.971447,71.428571,0.0


## Compose with Series operations

Kernels accept computed Series as inputs. Trivial arithmetic stays in Polars while the stateful rolling calculation stays in the kernel.

In [5]:
typical_price = (high + low + close) / 3.0
cci = kernels.cci(typical_price, period=20)

pl.DataFrame(
    {
        "date": prices["date"],
        "typical_price": typical_price,
        "cci20": cci,
    }
).tail()

date,typical_price,cci20
date,f64,f64
2024-08-05,206.256668,-217.254631
2024-08-06,206.09667,-182.707512
2024-08-07,209.950002,-126.059997
2024-08-08,212.113332,-93.109544
2024-08-09,214.996668,-56.807261


## Public API boundary

`bartons.indicators` is the supported user API: expressions compose with selection, grouping, lazy execution, and query optimization, and its catalog is complete regardless of implementation strategy. `bartons.kernels` is an uneven internal surface containing only indicators backed by Rust. Do not build application code against it; these eager bindings may be removed.